# Informativity exploratory tests

This file is mostly for testing the informativity Data-Driven Control (DDC) in the literature.

In [1]:
# Default packages
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import shutil
from scipy.constants import c, pi
from scipy import signal
from matplotlib import cm
from numpy.fft import fft, fftshift

# For nice plots
import plotly.io as pio
from plotly import subplots
import plotly.express as px
import plotly.graph_objects as go

# --- Dark/Interactive Theme ---
theme = "plotly_dark"

layout_defaults = dict(
    template=theme,
    paper_bgcolor="rgba(0,0,0,0)", # Transparent background
    plot_bgcolor="rgba(0,0,0,0)",  # Transparent plot area
    legend=dict(
        x=0.98,
        y=0.98,
        xanchor='right',
        yanchor='top',
        bgcolor="rgba(0, 0, 0, 0.4)",
        bordercolor="rgba(0,0,0,0)"
    ),
    xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.12)"),
    yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.12)"),
    width=650,
    height=325,
    autosize=False, # Prevents the browser from stretching it
    margin=dict(l=40, r=40, t=40, b=40),
    title = dict(
        x=0.5,
        xanchor='center'
    ),
    font = dict(
        family = "Helvetica",
        size = 14
    )
)

# Apply the defaults globally
pio.templates.default = theme
pio.templates[theme].layout.update(layout_defaults); # type: ignore

# For inline plots
%matplotlib inline

# This is for latex in plotly - if it doesn't work comment it
from IPython.display import display, HTML
display(HTML("<script src='https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.7/MathJax.js?config=TeX-MML-AM_CHTML'></script>"))

# Calc demonstration packages
import handcalcs.render
handcalcs.set_option("display_precision", 2) # type: ignore
import forallpeople as si
si.environment('default', top_level= True)

# Type defining for pylance
from typing import TYPE_CHECKING
# This block is only seen by Pylance/IDE
if TYPE_CHECKING:
    m = si.m                    # type: ignore
    kg = si.kg                  # type: ignore
    Ohm = si.Ohm                # type: ignore
    V = si.V                    # type: ignore
    A = si.A                    # type: ignore
    Hz = si.Hz                  # type: ignore
    W = si.W                    # type: ignore
    C = si.C                    # type: ignore
    F = si.F                    # type: ignore
    Wb = si.Wb                  # type: ignore
    T = si.T                    # type: ignore
    H = si.H                    # type: ignore
    Celsius = si.Celsius        # type: ignore
    lux = si.lux                # type: ignore
    s = si.s                    # type: ignore
    # Add any other units you use frequently

# Some unit naming conventions
ohm = Ohm

import cvxpy as cp
import control as ct

# Print all solvers available in your current environment
# Using clarabel for now
print(f"Listing installed solvers below:")
print(cp.installed_solvers())

# plt.rcParams.update(
#     {
#         "text.usetex": True,
#         "font.family": "typeface",
#     }
# )  # Makes plotting slower but prettier (requires LaTeX installed)

# For debugging personal modules
# %load_ext autoreload
# %autoreload 2

Listing installed solvers below:
['MOSEK', 'CLARABEL', 'SCS', 'SCIPY', 'HIGHS', 'OSQP']


## Example 6.3.2 - van Waarden's book

Right now, the code below only agrees qualitatively with the book. Numerically, there are significant differences, mainly the fact that, for $\epsilon = 0.5$ the solver only returns a 'good' result in 68/100 cases, whereas the reported result is 100/100.

In [10]:
# Testing packages with example 6.3.2 from van Waarde's book
#np.random.seed(67)  # For reproducibility
success_count = 0
stab_count = 0
N_exp = 100

# Running 100 times
for i in range(N_exp):
    # 1. Generate LTI System Data (python-control)
    Atrue = np.array([[0.850, -0.038, -0.38], 
                    [0.735, 0.815, 1.594],
                    [-0.664, 0.697, -0.064]])
    Btrue = np.array([[1.431, 0.705], 
                    [1.620, -1.129],
                    [0.913, 0.369]])

    n, m = Atrue.shape[0], Btrue.shape[1]

    # 2. Augment Input Matrix B_aug = [B, I_n] so noise w(k) enters directly as input
    Baug = np.hstack([Btrue, np.eye(n)])
    C = np.eye(n)

    # Simulate trajectory with input sequence U
    N = 21 # >= T+1
    U = np.random.randn(m, N)

    # Generate uniform direction vectors (Gaussian normalized)
    epsilon = 2.2
    raw_noise = np.random.normal(size=(n, N))
    norms = np.linalg.norm(raw_noise, axis=0, ord=2)
    unit_directions = raw_noise / norms

    # === NOTE === #
    # The way the sampling is done below doesn't correspon with the results in the book
    # Sample random radii up to sqrt(epsilon)
    radii = np.sqrt(epsilon) * (np.random.rand(1, N))
    W = unit_directions * radii  # Shape: (n, N), satisfying ||w(k)||_2^2 <= epsilon

    # This only almost works for epsilon = 0.5
    W = np.sqrt(epsilon)*np.random.randn(n, N)

    # Combine inputs: [u(k); w(k)] -> Shape: (m + n, N)
    U_augmented = np.vstack([U, W])

    # Runs sim and gathers data
    sys = ct.ss(Atrue, Baug, C, 0, dt=1.0)
    t_sim, X = ct.forced_response(sys, U=U_augmented, initial_state=np.random.randn(n)) # type: ignore

    Thor = 20
    Tinit = 0
    Xminus = X[:, Tinit:Thor+Tinit]  # X(k)
    Xplus  = X[:, Tinit+1:Thor+Tinit+1]  # X(k+1)
    Uminus = U[:, Tinit:Thor+Tinit]  # U(k)

    # 2. Informativity / LMI Formulation in CVXPY

    # Definition of Phi
    Phi_noise_bound = epsilon
    Phi11 = Thor*Phi_noise_bound*np.eye(n)
    Phi12 = np.zeros((n, Thor))
    Phi21 = Phi12.T
    Phi22 = -np.eye(Thor)

    # Phi
    Phi = np.block([[Phi11, Phi12], [Phi21, Phi22]])

    # Variables
    P = cp.Variable((n, n), PSD=True)
    L = cp.Variable((m, n))
    beta = cp.Variable(pos = True)

    # Theorem 6.25
    # Block 1
    lmi_m1 = cp.bmat(
        [
            [P - beta * np.eye(n), np.zeros((n, n)), np.zeros((n, m)), np.zeros((n, n))],
            [np.zeros((n, n)), -P, -L.T, np.zeros((n, n))],
            [np.zeros((m, n)), -L, np.zeros((m, m)), L],
            [np.zeros((n, n)), np.zeros((n, n)), L.T, P],
        ]
    )

    # Block 2
    lmi_data = np.block(
        [
            [np.eye(n), Xplus],
            [np.zeros((n, n)), -Xminus],
            [np.zeros((m, n)), -Uminus],
            [np.zeros((n, n)), np.zeros((n, Thor))],
        ]
    )

    # Block 2
    lmi_m2 = lmi_data @ Phi @ lmi_data.T

    # Full LMI
    tol = 1e-12
    constraints = [
        lmi_m1 - lmi_m2 >> 0,
        P >> tol*np.eye(n),
        beta >= tol
    ]
    prob = cp.Problem(cp.Minimize(1), constraints)
    # The clarabel options clearly need more refining
    # if mosek is an option, consider using it
    clarabel_opts = {
        'max_iter': int(1e5),
        'tol_gap_abs':1e-6,
        'tol_gap_rel':1e-6,
        'tol_feas':1e-8,
        'max_step_fraction':0.2,
        'iterative_refinement_max_iter': 20,
    }
    prob.solve(solver = 'MOSEK', verbose=False)

    # Extracts K and checks for stability
    if(prob.status == 'optimal'):
        success_count+=1

print(f"Success rate = {success_count/N_exp*100.0}%")

Success rate = 49.0%


## Example 8.2.1 from van Waarden's book

Checking to see if it yields the same result as the book

Issues:
    - Mosek doesn't run with default configs
    - SCS does run and returns a similar cost, though it's imprecise


In [184]:
# Example 8.2.1 - van Waarden

# System's matrices
Atrue = np.array(
    [
        [1.000, -0.374, -0.190, -0.321, 0.056, -0.026],
        [0.000, 0.982, 0.010, -0.000, -0.003, 0.001],
        [0.000, 0.115, 0.975, -0.000, -0.269, 0.191],
        [0.000, 0.001, 0.010, 1.000, -0.001, 0.001],
        [0.000, 0.000, 0.000, 0.000, 0.741, 0.000],
        [0.000, 0.000, 0.000, 0.000, 0.000, 0.741],
    ]
)

# B_true (2x6 transposed to 6x2 matrix)
Btrue = np.array(
    [
        [0.007, 0.000, -0.043, 0.000, 0.259, 0.000],
        [-0.003, 0.000, 0.030, 0.000, 0.000, 0.259],
    ]
).T

n, m = Atrue.shape[0], Btrue.shape[1]

Ctrue = np.eye(n)
Dtrue = 0.0

# Cost metric matrices
Ccost = np.array([[0, 0, 0, 0, 0, 1]])
Dcost = 0.0

# Dimension of z
nzcost = Ccost.shape[0]

# Augment Input Matrix B_aug = [B, I_n] so noise w(k) enters directly as input
Baug = np.hstack([Btrue, np.eye(n)])

# Sim. parameters
N = 751
sigma_w = 0.005

# Generates some signals for the sim

# Input
U = np.random.randn(m, N)

# Noise
W = sigma_w*np.random.randn(n, N)

# Combine inputs: [u(k); w(k)] -> Shape: (m + n, N)
Uaug = np.vstack([U, W])

# Runs sim and gathers data
sys = ct.ss(Atrue, Baug, Ctrue, 0, dt=1.0)
tsim, X = ct.forced_response(sys, U=Uaug, initial_state=np.random.randn(n)) # type: ignore

Thor = 750
Tinit = 0
Xminus = X[:, Tinit:Thor+Tinit]  # X(k)
Xplus  = X[:, Tinit+1:Thor+Tinit+1]  # X(k+1)
Uminus = U[:, Tinit:Thor+Tinit]  # U(k)

# 2. Informativity / LMI Formulation in CVXPY

# Definition of Phi
Phi_noise_bound = 1.35*sigma_w**2
Phi11 = Thor*Phi_noise_bound*np.eye(n)
Phi12 = np.zeros((n, Thor))
Phi21 = Phi12.T
Phi22 = -np.eye(Thor)

# Phi
Phi = np.block([[Phi11, Phi12], [Phi21, Phi22]])

# Mounts the LMIs in 8.30
# Variables
Y = cp.Variable((n, n), PSD=True)
Z = cp.Variable((n, n), symmetric=True)
L = cp.Variable((m, n))
alpha = cp.Variable(nonneg=True)
beta = cp.Variable(pos=True)
gamma = cp.Variable(pos=True)

# Matrix blocks
lmi_m1 = cp.bmat(
    [
        [Y - beta * np.eye(n), np.zeros((n, n)), np.zeros((n, m)), np.zeros((n, n)), np.zeros((n, nzcost))],
        [np.zeros((n, n)), np.zeros((n, n)), np.zeros((n, m)), Y, np.zeros((n, nzcost))],
        [np.zeros((m, n)), np.zeros((m, n)), np.zeros((m, m)), L, np.zeros((m, nzcost))],
        [np.zeros((n, n)), Y, L.T, Y, (Ccost@Y).T],
        [np.zeros((nzcost, n)), np.zeros((nzcost, n)), np.zeros((nzcost, m)), Ccost@Y, np.eye(nzcost)]
    ]
)

lmi_data = np.block(
    [
        [np.eye(n), Xplus],
        [np.zeros((n, n)), -Xminus],
        [np.zeros((m, n)), -Uminus],
        [np.zeros((n, n)), np.zeros((n, Thor))],
        [np.zeros((nzcost, n)), np.zeros((nzcost, Thor))]
    ]
)

lmi_m2 = lmi_data @ Phi @ lmi_data.T

lmi_m3 = cp.bmat(
    [
        [Y, (Ccost@Y).T],
        [Ccost@Y, np.eye(nzcost)]
    ]
)

lmi_m4 = cp.bmat(
    [
        [Z, np.eye(n)],
        [np.eye(n), Y]
    ]
)

# Full LMI
tol = 1e-12
constraints = [
    Y >> tol*np.eye(n),
    beta >= tol,
    gamma >= tol,
    lmi_m1 - alpha*lmi_m2 >> 0, # 1
    lmi_m3 >> tol*np.eye(n+nzcost), # 2
    lmi_m4 >> 0, # 3
    cp.trace(Z) <= gamma
]

# Define MOSEK specific tuning parameters
mosek_params = {
    'MSK_DPAR_OPTIMIZER_MAX_TIME':  100.0,
    'MSK_IPAR_INTPNT_SOLVE_FORM':   'MSK_SOLVE_DUAL',
}

prob = cp.Problem(cp.Minimize(gamma), constraints)
prob.solve(solver = 'MOSEK', verbose=True)
print(f"Status = {prob.status}")

(CVXPY) Aug 04 02:55:22 PM: Your problem has 87 variables, 673 constraints, and 0 parameters.
(CVXPY) Aug 04 02:55:22 PM: It is compliant with the following grammars: DCP, DQCP
(CVXPY) Aug 04 02:55:22 PM: DCP verification time: 0.0006 seconds.
(CVXPY) Aug 04 02:55:22 PM: Expression tree has 9 nodes.
(CVXPY) Aug 04 02:55:22 PM: (If you need to solve this problem multiple times, but with different data, consider using parameters.)
(CVXPY) Aug 04 02:55:22 PM: CVXPY will first compile your problem; then, it will invoke a numerical solver to obtain a solution.
(CVXPY) Aug 04 02:55:22 PM: Your problem is compiled with the CPP canonicalization backend.
(CVXPY) Aug 04 02:55:22 PM: Compiling problem (target solver=MOSEK).
(CVXPY) Aug 04 02:55:22 PM: Reduction chain: Dcp2Cone -> CvxAttr2Constr -> ExactCone2Cone -> EliminateZeroSized -> ConeMatrixStuffing -> MOSEK
(CVXPY) Aug 04 02:55:22 PM: Applying reduction Dcp2Cone
(CVXPY) Aug 04 02:55:22 PM: Applying reduction CvxAttr2Constr
(CVXPY) Aug 04 0

                                     CVXPY                                     
                                     v1.9.2                                    
-------------------------------------------------------------------------------
                                  Compilation                                  
-------------------------------------------------------------------------------
-------------------------------------------------------------------------------
                                Numerical solver                               
-------------------------------------------------------------------------------




SolverError: Solver 'MOSEK' failed. Try another solver, or solve with verbose=True for more information.

In [199]:
# debug

# Plotting sim data
fig = subplots.make_subplots(rows=3)
fig.add_trace(
    go.Scatter(
        y=U[0,:],
        mode="lines",
        name=r"$U(k)$",
    ),
    row=1,
    col=1
)
fig.add_trace(
    go.Scatter(
        y=X[5,:],
        mode="lines",
        name=r"$X(0)$",
    ),
    row=2,
    col=1
)
fig.add_trace(
    go.Scatter(
        y=np.linalg.norm(W, axis=0, ord=2),
        mode="lines",
    ),
    row=3,
    col=1
)

# Update axis labels and layout
#fig.update_yaxes(title_text="Voltage (V)", row=1)
#fig.update_yaxes(title_text="Voltage (V)", row=2)
#fig.update_xaxes(title_text="Time (s)", row=2)

fig.update_layout(
    height = 400
)

fig.show()
print(np.max((np.linalg.norm(W, axis=0, ord=2))**2))

0.000515050545212438
